# DUK leg kinematics

Interactive inspection of the reusable `duk_core` four-bar model.

**Frame convention used throughout this notebook:** robot `+Ox` points right, robot `+Oy` points up, counter-clockwise rotation is positive, and the directed vector **D→A is -30° from +Ox**. Therefore the opposite directed vector **A→D is +150°**. The solver keeps a separate local frame with A→D aligned to local +x; all robot-facing plots are transformed from that solver frame.


In [ ]:
from pathlib import Path
import math

from duk_core.modeling import jacobian_in_robot_frame, load_leg_geometry_json, solve_leg_pose, leg_jacobian, pose_in_robot_frame, sample_leg_trajectory
from duk_core.visualization import draw_leg_linkage, draw_point_trajectories, plot_point_coordinates, plot_point_jacobian

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'robot/config/physical/duk_leg_geometry.json').exists():
            return candidate
    raise FileNotFoundError('Run the notebook from a DUK repository checkout')

repo_root = find_repo_root()
geometry = load_leg_geometry_json(repo_root / 'robot/config/physical/duk_leg_geometry.json')
print(f'DA angle in robot frame = {math.degrees(geometry.da_angle_rad):+.1f} deg')
print(f'AD angle in robot frame = {math.degrees(geometry.ad_angle_rad):+.1f} deg')
geometry
print(f'physical assembly branch = {geometry.assembly_branch:+d}')


In [ ]:
lower, upper = geometry.geometric_angle_limits_rad()
print(f'BO = {geometry.bo_mm:.3f} mm')
print(f'CO = {geometry.co_mm:.3f} mm')
print(f'geometric angle range = {math.degrees(lower):.3f} .. {math.degrees(upper):.3f} deg')
print('The endpoints are singular boundaries, not operating limits.')


In [ ]:
branch = geometry.assembly_branch
input_angle_deg = 75.0  # Demo pose; this is not claimed to be the exact CAD snapshot angle.
solver_pose = solve_leg_pose(geometry, math.radians(input_angle_deg))
robot_pose = pose_in_robot_frame(geometry, solver_pose)
solver_jac = leg_jacobian(geometry, solver_pose)
robot_jac = jacobian_in_robot_frame(geometry, solver_jac)
print(f'input angle = {input_angle_deg:.2f} deg')
print(f'physical assembly branch = {branch:+d}')
print('A in robot frame =', robot_pose.a, 'mm')
print('D in robot frame =', robot_pose.d, 'mm')
print('D is higher than A:', robot_pose.d[1] > robot_pose.a[1])
print('B in robot frame =', robot_pose.b, 'mm')
print('C in robot frame =', robot_pose.c, 'mm')
print('O in robot frame =', robot_pose.o, 'mm')
print('dO/dalpha in robot frame =', robot_jac.d_o, 'mm/rad')
draw_leg_linkage(robot_pose, geometry).set_title(f'DUK leg linkage — physical branch {branch:+d}, demo angle {input_angle_deg:.1f}°')


In [ ]:
margin = math.radians(2.0)
solver_poses = sample_leg_trajectory(geometry, lower + margin, upper - margin, 180)
robot_poses = tuple(pose_in_robot_frame(geometry, pose) for pose in solver_poses)
draw_point_trajectories(robot_poses).set_title(f'B/C/H/O trajectories — physical branch {geometry.assembly_branch:+d}')


In [ ]:
plot_point_coordinates(robot_poses, 'O').set_title('Wheel-center coordinates — robot frame')


In [ ]:
plot_point_jacobian(geometry, solver_poses, 'O', rotation_rad=geometry.solver_to_robot_rotation_rad).set_title('Wheel-center Jacobian — robot frame')
